In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

In [2]:
path = "../DATASETS_SATTELITES/"

In [ ]:

# 1. CARREGAMENTO E LIMPEZA DE DADOS
def load_and_clean_data(file_path):
    # Ler os nomes das colunas da primeira linha (removendo o prefixo #)
    with open(file_path, 'r') as f:
        header = f.readline().lstrip('#').strip().split(',')
    
    # Carregar o dataset saltando a linha de atualização
    df = pd.read_csv(file_path, skiprows=2, names=header, comment='#', low_memory=False)

    # Lógica NAAN: Converter identificadores ausentes ou NAAN para string para evitar erros
    df['Satcat'] = df['Satcat'].astype(str).replace('nan', 'NAAN')

    # Conversão de Datas (LDate: Launch Date)
    # Extraímos os primeiros 10 caracteres para garantir o formato YYYY-MM-DD
    df['LDate_DT'] = pd.to_datetime(df['LDate'].str.slice(0, 10), errors='coerce')
    df['Year'] = df['LDate_DT'].dt.year

    # Conversão de Numéricos (Remover espaços e converter erros em NaN)
    cols_numeric = ['Mass', 'PERIGEE', 'APOGEE', 'INCLINATION']
    for col in cols_numeric:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

# Carregar os dados
df = load_and_clean_data(f'{path}/merged_dataset_tle.csv')

# --- ANÁLISE EXPLORATÓRIA COM PLOTLY ---

###########################################################
# 1. Evolução Temporal: Lançamentos por Ano e Tipo
###########################################################
df['DDate'] = pd.to_datetime(df['DDate'], errors='coerce')
df['SDate'] = pd.to_datetime(df['SDate'], errors='coerce')

    # We want DDate for debris (end of life) and SDate for active satellites (start of life)
df['Timeline_Date'] = df.apply(
    lambda row: row['DDate'] if row['Type'] == 'Debris' else row['SDate'], 
    axis=1
)
df['YearMonth'] = df['Timeline_Date'].dt.to_period('M').dt.to_timestamp()
monthly_data = df.groupby(['YearMonth', 'Type']).size().reset_index(name='Count')
monthly_data = monthly_data.sort_values('YearMonth')

    ############################################################
    # Only Active (Satellites, Space Stations, Components)
    ############################################################
df_active = monthly_data[monthly_data['Type'] != 'Debris']

fig_active = px.line(df_active, x='YearMonth', y='Count', color='Type',
                    title='Launches: Satellites, Space Stations and Components (SDate)',
                    labels={'Count': 'Launch Quantity', 'YearMonth': 'Date'},
                    template='plotly_dark')
fig_active.show()

    ############################################################
    # Only Debris
    ############################################################
df_debris = monthly_data[monthly_data['Type'] == 'Debris']

fig_debris = px.area(df_debris, x='YearMonth', y='Count',
                    title='Debris Fragmentation Evolution (DDate)',
                    labels={'Count': 'New Debris', 'YearMonth': 'Detach Date'},
                    color_discrete_sequence=['#ef553b'], # Cor de alerta/perigo
                    template='plotly_dark')
fig_debris.show()
############################################################
# 2. Geopolítica: Top 10 Países/Entidades
############################################################
top_states = df['State'].value_counts().head(10).reset_index()
fig2 = px.bar(top_states, x='count', y='State', orientation='h',
             title='Top 10 Entidades com Mais Objetos em Órbita',
             labels={'count': 'Total de Objetos', 'State': 'Estado/Organização'},
             color='count', color_continuous_scale='Viridis',
             template='plotly_white')
fig2.show()

# 3. Dinâmica Orbital: Distribuição por Regime de Órbita (OpOrbit)
orbit_counts = df['OpOrbit'].value_counts().head(12).reset_index()
fig3 = px.pie(orbit_counts, values='count', names='OpOrbit', 
             title='Distribuição por Regimes Orbitais (LEO, GEO, MEO, etc.)',
             hole=0.4, template='plotly_white')
fig3.show()

# 4. Densidade de Satélites: Altitude vs Inclinação
# Filtramos LEO (Baixa órbita < 2000km) para melhor visualização
leo_df = df[df['PERIGEE'] < 2000].dropna(subset=['INCLINATION', 'PERIGEE'])
fig4 = px.scatter(leo_df.sample(n=min(15000, len(leo_df))), # Amostra para performance
                 x='PERIGEE', y='INCLINATION', color='Type',
                 opacity=0.4, size_max=5,
                 title='Densidade em LEO: Altitude de Perigeu vs Inclinação',
                 labels={'PERIGEE': 'Altitude (km)', 'INCLINATION': 'Inclinação (graus)'},
                 template='plotly_dark')
fig4.show()

# 5. Análise de Massa: Evolução do Tamanho dos Objetos
mass_trend = df.groupby('Year')['Mass'].median().reset_index()
fig5 = px.scatter(mass_trend, x='Year', y='Mass', trendline="lowess",
                 title='Evolução da Massa Mediana dos Satélites (Miniaturização vs Grandes Constelações)',
                 labels={'Mass': 'Massa Mediana (kg)'},
                 template='plotly_white')
fig5.show()

# 6. O Fenómeno SpaceX (Análise de Propriedade)
spacex_df = df[df['Owner'] == 'SPXS'].groupby('Year').size().reset_index(name='Count')
fig6 = px.area(spacex_df, x='Year', y='Count', 
              title='Impacto da SpaceX (Starlink) no Volume de Lançamentos',
              labels={'Count': 'Objetos Lançados pela SpaceX'},
              color_discrete_sequence=['#005288'])
fig6.show()

# MÉTRICA EXTRA: Cálculo de Lifetime (Tempo em órbita)
df['DDate_DT'] = pd.to_datetime(df['DDate'].str.slice(0, 10), errors='coerce')
df['Lifetime_Days'] = (df['DDate_DT'] - df['LDate_DT']).dt.days

# Visualizar Lifetime para objetos que já reentraram
lifetime_df = df[df['Lifetime_Days'] > 0]
fig7 = px.histogram(lifetime_df, x='Lifetime_Days', nbins=50,
                   title='Distribuição do Tempo de Vida de Objetos que Reentraram',
                   labels={'Lifetime_Days': 'Dias em Órbita'},
                   log_y=True, template='plotly_white')
fig7.show()

C:\Users\Xaneca\AppData\Local\Temp\ipykernel_8896\1655639119.py:15: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

C:\Users\Xaneca\AppData\Local\Temp\ipykernel_8896\1655639119.py:31: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



KeyError: 'State'